# Testing Agents(nodes)

- This notebook tests the functionality of all nodes 
- including llm(API key working or not)

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"e:\SSPL_Internship_Repo\Jan_05_26\Trip_Event_Planner")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: e:\SSPL_Internship_Repo\Jan_05_26\Trip_Event_Planner


In [2]:
from config import setup_logger, settings

logger = setup_logger("Notebook-Testing-Agents(Nodes)")

logger.info("Logger initialized for testing agents notebook.")

logger.info(f"Groq API key loaded: {bool(settings.groq_api_key)}")
logger.info(f"Groq model: {settings.groq_model_name}")

2026-01-08 15:38:56 || INFO || Notebook-Testing-Agents(Nodes) || Logger initialized for testing agents notebook.
2026-01-08 15:38:56 || INFO || Notebook-Testing-Agents(Nodes) || Groq API key loaded: True
2026-01-08 15:38:56 || INFO || Notebook-Testing-Agents(Nodes) || Groq model: llama-3.3-70b-versatile


In [3]:
from langchain_core.messages import HumanMessage
from nodes import intent_node, research_node, planner_node, pricing_node, review_node, final_presenter_node

2026-01-08 15:38:58 || INFO || llm || Initializing Groq LLM
2026-01-08 15:38:59 || INFO || llm || Groq LLM initialized | model=llama-3.3-70b-versatile | temperature=0.3


In [4]:
state = {
    "messages": [
        HumanMessage(
            content="I want to plan a birthday trip to Goa for 4 people between January 10 and January 15 with a budget of 80k"
        )
    ]
}

state

{'messages': [HumanMessage(content='I want to plan a birthday trip to Goa for 4 people between January 10 and January 15 with a budget of 80k', additional_kwargs={}, response_metadata={})]}

In [5]:
state_update = intent_node(state)
state.update(state_update)

state

2026-01-08 15:39:03 || INFO || nodes.intent || Running intent extraction node
2026-01-08 15:39:04 || INFO || nodes.intent || Extracted user intent successfully


{'messages': [HumanMessage(content='I want to plan a birthday trip to Goa for 4 people between January 10 and January 15 with a budget of 80k', additional_kwargs={}, response_metadata={})],
 'user_intent': UserIntentModel(destination='Goa', start_date='2024-01-10', end_date='2024-01-15', budget=80000, event_type='birthday', group_size=4, preferences=[]),
 'current_stage': 'research'}

In [6]:
state_update = research_node(state)
state.update(state_update)

state.keys()

2026-01-08 15:39:09 || INFO || nodes.research || Running research agent
2026-01-08 15:39:09 || INFO || tools.research || Searching destination info: Goa
2026-01-08 15:39:10 || INFO || tools.research || Searching weather info: Weather in Goa during 2024-01-10 to 2024-01-15
2026-01-08 15:39:12 || INFO || tools.research || Searching venues: Best venues for birthday in Goa
2026-01-08 15:39:13 || INFO || tools.research_cost || Extracting flight price range
2026-01-08 15:39:13 || INFO || tools.research || Searching destination info: average flight cost to Goa
2026-01-08 15:39:15 || INFO || tools.research_cost || Extracted flight price range:{'category': 'flight', 'min_price': 33, 'max_price': 709, 'currency': 'USD', 'confidence': 'high'}
2026-01-08 15:39:15 || INFO || tools.research_cost || Extracting hotel price range
2026-01-08 15:39:15 || INFO || tools.research || Searching destination info: average hotel cost per night in Goa
2026-01-08 15:39:17 || INFO || tools.research_cost || Extracte

dict_keys(['messages', 'user_intent', 'current_stage', 'research_result'])

In [7]:
research = state["research_result"]

for key, value in research.model_dump().items():
    print(f"{key}: {value}")

confirmed_destination: Goa
weather_summary: Generally clear and sunny with temperatures ranging from 21°C to 32°C
best_travel_window: January, considering the weather conditions
venue_options: [{'name': 'Synergy Ballroom at Ginger Goa Candolim', 'venue_type': 'ballroom', 'estimated_price_per_day': None, 'suitable_for_event': True}, {'name': 'Grand Ballroom at Hilton Goa Resort', 'venue_type': 'ballroom', 'estimated_price_per_day': None, 'suitable_for_event': True}, {'name': 'Casa Sarita Restaurant of ITC Grand Goa Resort and Spa', 'venue_type': 'restaurant', 'estimated_price_per_day': None, 'suitable_for_event': True}, {'name': 'Masala Restaurant of ITC Grand Goa Resort and Spa', 'venue_type': 'restaurant', 'estimated_price_per_day': None, 'suitable_for_event': True}, {'name': 'Praia de Luz Restaurant of ITC Grand Goa Resort and Spa', 'venue_type': 'restaurant', 'estimated_price_per_day': None, 'suitable_for_event': True}, {'name': 'Cabana Private Restaurants and Bars of ITC Grand Goa 

In [8]:
research_review_fn = review_node("research")

state_update = research_review_fn(state)
state.update(state_update)

state["research_review"]

2026-01-08 15:39:33 || INFO || nodes.review || Entering review node for stage: research



REVIEW STAGE: RESEARCH

Research Summary:
confirmed_destination='Goa' weather_summary='Generally clear and sunny with temperatures ranging from 21°C to 32°C' best_travel_window='January, considering the weather conditions' venue_options=[VenueOptionModel(name='Synergy Ballroom at Ginger Goa Candolim', venue_type='ballroom', estimated_price_per_day=None, suitable_for_event=True), VenueOptionModel(name='Grand Ballroom at Hilton Goa Resort', venue_type='ballroom', estimated_price_per_day=None, suitable_for_event=True), VenueOptionModel(name='Casa Sarita Restaurant of ITC Grand Goa Resort and Spa', venue_type='restaurant', estimated_price_per_day=None, suitable_for_event=True), VenueOptionModel(name='Masala Restaurant of ITC Grand Goa Resort and Spa', venue_type='restaurant', estimated_price_per_day=None, suitable_for_event=True), VenueOptionModel(name='Praia de Luz Restaurant of ITC Grand Goa Resort and Spa', venue_type='restaurant', estimated_price_per_day=None, suitable_for_event=True)

2026-01-08 15:39:36 || INFO || nodes.review || Review decision | stage=research | approved=True


ReviewDecisionModel(approved=True, feedback=None, revision_target=None)

In [9]:
state_update = planner_node(state)
state.update(state_update)

state["itinerary"]

2026-01-08 15:39:40 || INFO || nodes.planner || Running planner agent
2026-01-08 15:39:40 || INFO || tools.planner || Calculated trip days: 6 days
2026-01-08 15:39:41 || INFO || nodes.planner || Itinerary created successfully


ItineraryModel(total_days=6, days=[DayPlanModel(day_number=1, date='2024-01-10', activities=['Arrival in Goa', 'Check-in at hotel'], is_event_day=False, notes=None), DayPlanModel(day_number=2, date='2024-01-11', activities=['Visit to Calangute Beach', 'Water sports activities'], is_event_day=False, notes=None), DayPlanModel(day_number=3, date='2024-01-12', activities=['Visit to Fort Aguada', 'Dinner at Thalassa'], is_event_day=False, notes=None), DayPlanModel(day_number=4, date='2024-01-13', activities=['Birthday celebration at Synergy Ballroom at Ginger Goa Candolim', 'Cake cutting ceremony'], is_event_day=True, notes=None), DayPlanModel(day_number=5, date='2024-01-14', activities=['Visit to Baga Beach', 'Shopping at local markets'], is_event_day=False, notes=None), DayPlanModel(day_number=6, date='2024-01-15', activities=['Check-out from hotel', 'Departure from Goa'], is_event_day=False, notes=None)], event_day=4, event_details='Birthday celebration at Synergy Ballroom at Ginger Goa 

In [10]:
planner_review_fn = review_node("planner")

state_update = planner_review_fn(state)
state.update(state_update)

state["planner_review"]


2026-01-08 15:39:44 || INFO || nodes.review || Entering review node for stage: planner



REVIEW STAGE: PLANNER

Itinerary:
total_days=6 days=[DayPlanModel(day_number=1, date='2024-01-10', activities=['Arrival in Goa', 'Check-in at hotel'], is_event_day=False, notes=None), DayPlanModel(day_number=2, date='2024-01-11', activities=['Visit to Calangute Beach', 'Water sports activities'], is_event_day=False, notes=None), DayPlanModel(day_number=3, date='2024-01-12', activities=['Visit to Fort Aguada', 'Dinner at Thalassa'], is_event_day=False, notes=None), DayPlanModel(day_number=4, date='2024-01-13', activities=['Birthday celebration at Synergy Ballroom at Ginger Goa Candolim', 'Cake cutting ceremony'], is_event_day=True, notes=None), DayPlanModel(day_number=5, date='2024-01-14', activities=['Visit to Baga Beach', 'Shopping at local markets'], is_event_day=False, notes=None), DayPlanModel(day_number=6, date='2024-01-15', activities=['Check-out from hotel', 'Departure from Goa'], is_event_day=False, notes=None)] event_day=4 event_details='Birthday celebration at Synergy Ballro

2026-01-08 15:39:47 || INFO || nodes.review || Review decision | stage=planner | approved=True


ReviewDecisionModel(approved=True, feedback=None, revision_target=None)

In [11]:
state_update = pricing_node(state)
state.update(state_update)

state["pricing"]

2026-01-08 15:39:53 || INFO || nodes.pricing || Running pricing agent
2026-01-08 15:39:53 || INFO || tools.pricing || Calculated total cost: 9801
2026-01-08 15:39:53 || INFO || tools.pricing || Applied buffer: 980
2026-01-08 15:39:53 || INFO || tools.pricing || Identified cost risks: []
2026-01-08 15:39:54 || INFO || nodes.pricing || Pricing completed successfully


PricingModel(cost_breakdown=CostBreakdownModel(flights=371, accommodation=3990, local_transport=3600, event_cost=1840, buffer=980), total_estimated_cost=10781, risk_factors=[], cost_saving_options=['Consider budget airlines for flights', 'Look for discounted accommodation options', 'Research local transport alternatives'], confidence_level='medium')

In [12]:
pricing_review_fn = review_node("pricing")

state_update = pricing_review_fn(state)
state.update(state_update)

state["pricing_review"]

2026-01-08 15:42:13 || INFO || nodes.review || Entering review node for stage: pricing



REVIEW STAGE: PRICING

Pricing Details:
cost_breakdown=CostBreakdownModel(flights=371, accommodation=3990, local_transport=3600, event_cost=1840, buffer=980) total_estimated_cost=10781 risk_factors=[] cost_saving_options=['Consider budget airlines for flights', 'Look for discounted accommodation options', 'Research local transport alternatives'] confidence_level='medium'

--- Approval ---


2026-01-08 15:42:16 || INFO || nodes.review || Review decision | stage=pricing | approved=True


ReviewDecisionModel(approved=True, feedback=None, revision_target=None)

In [13]:
from states.schemas.final_output import FinalTripPlanModel

state["final_output"] = FinalTripPlanModel(
    user_intent=state["user_intent"],
    research_summary=state["research_result"],
    itinerary=state["itinerary"],
    pricing=state["pricing"],
)

state_update = final_presenter_node(state)
state.update(state_update)

state["user_friendly_output"]


2026-01-08 15:42:20 || INFO || nodes.final_presenter || Running final presenter node
2026-01-08 15:42:22 || INFO || nodes.final_presenter || Final user-friendly output generated


UserFriendlyTripPlan(title='Your Personalized Trip & Event Plan 🎉', summary="Celebrating a Birthday in Goa\n\nWe've planned a 6-day trip to Goa for your birthday celebration from January 10th to January 15th. You'll be traveling with a group of 4, and we've tailored the itinerary to ensure a memorable experience for everyone.\n\nYour trip will begin with an arrival in Goa and check-in at a hotel. The following days will be filled with exciting activities such as visiting Calangute Beach, trying water sports, exploring Fort Aguada, and enjoying dinner at Thalassa. The highlight of your trip will be the birthday celebration at the Synergy Ballroom at Ginger Goa Candolim, complete with a cake-cutting ceremony.\n\nOn other days, you'll have the opportunity to visit Baga Beach, go shopping at local markets, and relax at your hotel. Your trip will come to an end with a check-out from the hotel and departure from Goa.\n\nIn terms of budget, we've estimated a total cost of approximately ₹78,00

In [14]:
final_op = state["user_friendly_output"]
for key, value in final_op.model_dump().items():
    print(f"{key}: {value}")

title: Your Personalized Trip & Event Plan 🎉
summary: Celebrating a Birthday in Goa

We've planned a 6-day trip to Goa for your birthday celebration from January 10th to January 15th. You'll be traveling with a group of 4, and we've tailored the itinerary to ensure a memorable experience for everyone.

Your trip will begin with an arrival in Goa and check-in at a hotel. The following days will be filled with exciting activities such as visiting Calangute Beach, trying water sports, exploring Fort Aguada, and enjoying dinner at Thalassa. The highlight of your trip will be the birthday celebration at the Synergy Ballroom at Ginger Goa Candolim, complete with a cake-cutting ceremony.

On other days, you'll have the opportunity to visit Baga Beach, go shopping at local markets, and relax at your hotel. Your trip will come to an end with a check-out from the hotel and departure from Goa.

In terms of budget, we've estimated a total cost of approximately ₹78,000 for the entire trip, which in

In [15]:
logger.info("Completed testing of agents notebook.")

2026-01-08 15:43:14 || INFO || Notebook-Testing-Agents(Nodes) || Completed testing of agents notebook.
